# HF Export & Web UI

Convert your trained DustyLM checkpoint to ONNX, push it to Hugging Face Hub, and serve the browser-based chat UI. Run this after you have a promoted SFT checkpoint from the [training notebook](02_train_from_scratch.ipynb).

---
## Setup

In [ ]:
# ── Environment Setup ────────────────────────────────────────────────────
# Colab: clones the repo and installs dependencies.
# Local / RunPod: assumes you are in the repo root with deps installed.

import sys
from pathlib import Path

if "google.colab" in sys.modules:
    !git clone https://github.com/mkhordoo/dusty-lm.git
    %cd dusty-lm
    !pip install -q uv && uv pip install --system -e '.[onnx]'
    print("\u2713 Colab environment ready (with ONNX support).")
elif not Path("dustylm").exists():
    raise RuntimeError(
        "Run this notebook from the repository root.\n"
        "  cd /path/to/dusty-lm && jupyter notebook"
    )
else:
    print("\u2713 Running from repo root. Dependencies assumed installed.")
    print("  For ONNX export, ensure you have: pip install -e '.[onnx]'")

---

## Serve the Browser Demo Locally

The best way to interact with your newly trained model is through the browser.

After you promote a checkpoint, run `make serve-web` to start the local browser demo. By default it serves the latest promoted `sft_dusty8m` checkpoint at `http://localhost:8000`, so open that URL in your browser after the command starts.

To try a specific SFT checkpoint without promoting it first, pass `CHECKPOINT_STEP`. If port 8000 is already in use, override it with `WEB_PORT`, then open the matching localhost URL. The target refreshes the browser files before starting the local server.

In [ ]:
# Serve the promoted checkpoint locally.
!make serve-web

# Or serve a specific SFT checkpoint step directly.
# !make serve-web CHECKPOINT_STEP=1000

# Or use a different local port, then open http://localhost:9000.
# !make serve-web WEB_PORT=9000

*(Behind the scenes, this command automatically converts your PyTorch weights into an optimized ONNX format so it can run lightning-fast directly in your browser!)*

---

## Export to ONNX (Optional / Advanced)

If you want to take your trained model and deploy it to a mobile app, an edge device, or a custom backend, you can manually export it to the ONNX format. ONNX is a highly optimized, universally supported machine learning standard.

In [ ]:
!make dusty-export-onnx ONNX_PROFILE=sft_dusty8m

---

## Push to Hugging Face Hub

The Hub upload flow is intentionally staged. It does **not** read from `docs/` and it does **not** upload one file at a time. The script rebuilds a clean local folder under `artifacts/hub_upload/<profile>/`, then `upload_folder` pushes that exact folder as one Hugging Face commit.

Before a live push:

- Create the target model repo on Hugging Face, such as `your-username/dusty-8m-sft`.
- Log in with `huggingface-cli login` or set `HF_TOKEN` in your environment.
- Edit `artifacts/hf/HF_MODEL_CARD.md`; the script copies it to `README.md` for the Hub.
- Make sure `artifacts/checkpoints/dusty8m_sft.pt`, `artifacts/tokenizers/dusty_tokenizer.json`, and `docs/images/logo.png` exist.

The staged Hub folder includes `model.pt`, a freshly exported `model_int8.onnx`, tokenizer files with the Hugging Face chat template, `README.md`, and `assets/logo.png` for the model card image.

### Dry Run: Build the Hub Folder Locally

Start here. This rebuilds the staging folder and stops before upload, so you can inspect the exact files that would be pushed.

In [ ]:
# Change this to your Hugging Face model repo.
HF_REPO_ID = "your-username/dusty-8m-sft"

!make stage-hub HF_REPO_ID={HF_REPO_ID}

In [ ]:
from pathlib import Path

staging_dir = Path("artifacts/hub_upload/sft_dusty8m")
for path in sorted(
    p.relative_to(staging_dir) for p in staging_dir.rglob("*") if p.is_file()
):
    print(path)

### Live Push to Hugging Face Hub

Run this only after the dry-run folder looks correct and your Hugging Face authentication is configured. If the upload fails with an authentication error, set `HF_TOKEN` or run `huggingface-cli login`, then rerun the command.

In [ ]:
# Live upload. This pushes the staged model folder to the Hugging Face Hub repo.
!make push-hub HF_REPO_ID={HF_REPO_ID}